## Mount drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Import packages/install libraries

In [ ]:
!pip install segmentation-models-pytorch

In [ ]:
import cv2
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
import torchvision.transforms as T

import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp

## Data Preprocessing

In [ ]:
# Using GPU cuz CPU takes too damn long
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Define transformation

# Add augmentation for training
train_transformation = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),
    A.Affine(translate_percent=0.05, scale=(0.9, 1.1), rotate=0, p=0.5),
    A.RandomBrightnessContrast(p=0.3),
    A.GaussNoise(p=0.2),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

transformation = A.Compose([
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

### Define Dataset Class

In [ ]:
def lesion_focused_crop(image, mask, crop_size=(256, 256)):
  h, w = mask.shape
  ch, cw = crop_size

  # Find lesion pixels (assuming lesion > 0)
  lesion_pixels = np.argwhere((mask != 255) & (mask > 0))

  if len(lesion_pixels) == 0:
    # fallback to random crop if no lesion
    y = np.random.randint(0, max(1, h - ch + 1))
    x = np.random.randint(0, max(1, w - cw + 1))
  else:
    # pick random lesion pixel
    y, x = lesion_pixels[np.random.randint(len(lesion_pixels))]

    # center crop around it
    y = np.clip(y - ch // 2, 0, max(0, h - ch))
    x = np.clip(x - cw // 2, 0, max(0, w - cw))

  cropped_img = image[y:y+ch, x:x+cw]
  cropped_mask = mask[y:y+ch, x:x+cw]

  return cropped_img, cropped_mask

In [ ]:
# Define dataset class to apply preprocessing and handle masks
class GingivitisDataset(Dataset):
  def __init__(self, image_folder, mask_folder, transform=None):
    self.image_folder = Path(image_folder)
    self.mask_folder = Path(mask_folder)
    self.transform = transform

    # Get all images
    self.image_paths = sorted(self.image_folder.glob("*.jpg"))

    # Define a simple colour-to-class mapping
    self.colour_to_label = {
      (0, 255, 0):    0,    # healthy
      (255, 255, 0): 1,    # mild
      (255, 165, 0):  2,    # moderate
      (255, 0, 0):  3,    # severe
      (139, 0, 0):    4,    # very severe
      (0, 0, 0):      255,  # background   - ignore during training
    }

    # Match masks directly
    self.mask_paths = [
        self.mask_folder / f"{img_path.stem}.png"
        for img_path in self.image_paths
    ]

  # Total number of images
  def __len__(self):
    return len(self.image_paths)

  def __getitem__(self, idx):
    # Load image
    img_path = self.image_paths[idx]
    mask_path = self.mask_paths[idx]
    image_name = img_path.name

    image = cv2.imread(str(img_path))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Load mask
    mask = cv2.imread(str(mask_path))

    # Check missing masks
    if mask is None:
      raise FileNotFoundError(f"Mask file not found or could not be loaded for image: {img_path}. Expected mask path: {mask_path}")

    mask = cv2.cvtColor(mask, cv2.COLOR_BGR2RGB)
    mask = mask.astype(np.uint8)

    # Convert colours to labels
    mask_label = np.full((mask.shape[0], mask.shape[1]), 255, dtype=np.uint8) # np.full defaults to 255 so unrecognised colours are ignored

    tol = 15

    # Add tolerance matching
    for colour, label in self.colour_to_label.items():
      target = np.array(colour)
      matches = np.all(np.abs(mask - target) <= tol, axis=-1)
      mask_label[(matches) & (mask_label == 255)] = label

    if not np.any(mask_label != 255):
      print(f"Warning: No valid labels found in {image_name}")

    # Balanced cropping
    if np.random.rand() < 0.7:
      image, mask_label = lesion_focused_crop(image, mask_label)
    else:
      h, w = mask_label.shape
      ch, cw = 256, 256
      y = np.random.randint(0, max(1, h - ch + 1))
      x = np.random.randint(0, max(1, w - cw + 1))

      image = image[y:y+ch, x:x+cw]
      mask_label = mask_label[y:y+ch, x:x+cw]

    # Apply transform
    if self.transform:
      augmented = self.transform(image=image, mask=mask_label)
      image = augmented["image"]
      mask_label  = augmented["mask"].long()

    return image, mask_label, image_name

### Create dataset objects

In [ ]:
train_dataset = GingivitisDataset(
    image_folder="/content/drive/MyDrive/CSS2/CSS2_480p/Dataset/Training/Images",
    mask_folder="/content/drive/MyDrive/CSS2/CSS2_480p/Dataset/Training/Masks",
    transform=train_transformation
)

val_dataset = GingivitisDataset(
    image_folder="/content/drive/MyDrive/CSS2/CSS2_480p/Dataset/Validation/Images",
    mask_folder="/content/drive/MyDrive/CSS2/CSS2_480p/Dataset/Validation/Masks",
    transform=transformation
)

test_dataset = GingivitisDataset(
    image_folder="/content/drive/MyDrive/CSS2/CSS2_480p/Dataset/Test/Images",
    mask_folder="/content/drive/MyDrive/CSS2/CSS2_480p/Dataset/Test/Masks",
    transform=transformation
)

## Create DataLoaders

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=2, pin_memory=True) # shuffle is true for training so model learns instead of memorising; pin_memory is to transfer from cpu to gpu faster
val_loader   = DataLoader(val_dataset, batch_size=4, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset, batch_size=4, shuffle=False, num_workers=2, pin_memory=True)

## Load UNet

In [ ]:
model = smp.Unet(
    encoder_name="efficientnet-b4",    # encoder backbone
    encoder_weights="imagenet",        # pretrained weights on ImageNet
    in_channels=3,                     # RGB channels/images
    classes=5                          # 5 classes (healthy to very severe); background is ignored
)

model = model.to(device) # using GPU

config.json:   0%|          | 0.00/106 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/77.9M [00:00<?, ?B/s]

In [ ]:
# Loss function and optimizer
criterion = torch.nn.CrossEntropyLoss(ignore_index=255)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

## Training

In [ ]:
num_epochs = 10

for epoch in range(num_epochs):
  # Training loop
  model.train()
  total_loss = 0
  train_batches = 0

  for i, (images, masks, names) in enumerate(train_loader):
    images = images.to(device)   # using GPU
    masks = masks.to(device)

    # Skip batch if there are no valid pixels in the mask for loss calculation
    if not (masks != 255).any():
      print(f"Training Skipping Batch {i}, images: {names} since all mask pixels are ignore_index (255)")
      continue

    outputs = model(images)
    loss = criterion(outputs, masks)

    optimizer.zero_grad() # clear old gradients from previous steps
    loss.backward() # backward pass to calculate the gradients for all parameters
    optimizer.step() # update model

    total_loss += loss.item()
    train_batches += 1

  # Validation loop
  model.eval() # set model to evaluation mode
  val_loss = 0
  val_batches = 0

  with torch.no_grad():
    for i, (images, masks, names) in enumerate(val_loader):
      images = images.to(device)
      masks = masks.to(device)

      # Skip batch if there are no valid pixels in the mask for loss calculation
      if not (masks != 255).any():
        print(f"Validation Skipping Batch {i}, images: {names} since all mask pixels are ignore_index (255).")
        continue # Skip this batch

      outputs = model(images)
      loss = criterion(outputs, masks)
      val_loss += loss.item()
      val_batches += 1

  train_loss = total_loss / train_batches if train_batches > 0 else 0
  val_loss = val_loss / val_batches if val_batches > 0 else 0

  print(f"Epoch {epoch+1}")
  print(f"Train loss: {train_loss:.2f}")
  print(f"Val loss: {val_loss:.2f}")

Epoch 1
Train loss: 0.97
Val loss: 1.38
Epoch 2
Train loss: 0.93
Val loss: 1.36
Epoch 3
Train loss: 0.92
Val loss: 1.36
Epoch 4
Train loss: 0.92
Val loss: 1.38
Epoch 5
Train loss: 0.91
Val loss: 1.31
Epoch 6
Train loss: 0.92
Val loss: 1.33
Epoch 7
Train loss: 0.92
Val loss: 1.35
Epoch 8
Train loss: 0.90
Val loss: 1.35
Epoch 9
Train loss: 0.92
Val loss: 1.33
Epoch 10
Train loss: 0.90
Val loss: 4.86


## Testing

IoU checks how much the prediction overlaps with the ground truth (correct labels).  
Dice measures similarity between prediction and ground truth but gives more weight to overlapping pixels.  
Calculating the mean for these two to show one score overall for them.

In [ ]:
def evaluate(model, dataloader, device, num_classes=5):
  model.eval()

  total_correct = 0
  total_pixels = 0

  # Accumulate per-class stats (dataset-level)
  intersection_per_class = [0] * num_classes
  union_per_class = [0] * num_classes
  dice_num = [0] * num_classes
  dice_den = [0] * num_classes

  with torch.no_grad():
    for images, masks, names in dataloader:
      images = images.to(device)
      masks = masks.to(device)

      outputs = model(images)
      preds = torch.argmax(outputs, dim=1)

      valid = masks != 255  # ignore background

      # Accuracy
      correct = (preds == masks) & valid
      total_correct += correct.sum().item()
      total_pixels += valid.sum().item()

      # Per-class metrics
      for cls in range(num_classes):
        pred_cls = (preds == cls) & valid
        mask_cls = (masks == cls) & valid

        intersection = (pred_cls & mask_cls).sum().item()
        union = (pred_cls | mask_cls).sum().item()

        intersection_per_class[cls] += intersection
        union_per_class[cls] += union

        dice_num[cls] += 2 * intersection
        dice_den[cls] += pred_cls.sum().item() + mask_cls.sum().item()

  # Safe accuracy
  accuracy = total_correct / total_pixels if total_pixels > 0 else 0

  # Mean IoU (dataset-level)
  iou_scores = []
  for c in range(num_classes):
    if union_per_class[c] > 0:
      iou_scores.append(intersection_per_class[c] / union_per_class[c])

  mean_iou = sum(iou_scores) / len(iou_scores) if iou_scores else 0

  # Mean Dice (dataset-level)
  dice_scores = []
  for c in range(num_classes):
    if dice_den[c] > 0:
        dice_scores.append(dice_num[c] / (dice_den[c] + 1e-6))

  mean_dice = sum(dice_scores) / len(dice_scores) if dice_scores else 0

  return accuracy, mean_iou, mean_dice

In [ ]:
test_acc, test_iou, test_dice = evaluate(model, test_loader, device)

print(f"Test Accuracy: {test_acc:.2f}")
print(f"Mean IoU: {test_iou:.2f}")
print(f"Mean Dice: {test_dice:.2f}")

Test Accuracy: 0.10
Mean IoU: 0.03
Mean Dice: 0.06
